# 02 · Retrieval evaluation: BM25 NDCG@k for Original / UFD / SFD

Reproduces the paper's keyword-retrieval evaluation with AutoDDG's own code:
for every query we index **only that query's candidate pool** with
`autoddg.search.SimpleSearchEngine("BM25")`, rank the whole pool with `search(query, size=len(pool))`, and score the
ranking with `autoddg.ranking.compute_ndcg` (gain 2^rel − 1, graded rel ∈ {0,1,2}) at k = 5, 10, 15, 20.

* **Original** needs only `benchmark/` (built by notebook 01, section 3), so run it first as a sanity check of the
  benchmark itself. The paper reports **0.688** NDCG@20 for BM25 on original descriptions.
* **UFD / SFD** come from `descriptions.jsonl` produced on the Mac (`mac/generate_descriptions.py`). The paper reports
  **0.705** NDCG@20 for AutoDDG-SFD (GPT-4o-mini).

Runs on Colab (reads Drive) or locally (point `WORK_DIR` at an unzipped `ntcir_ddg_intermediate.zip`).

## Configuration

In [ ]:
import os

WORK_DIR = os.environ.get("NTCIR_DDG_WORK_DIR", "/content/drive/MyDrive/ntcir_ddg")
DESCRIPTIONS_PATH = os.environ.get("NTCIR_DDG_DESCRIPTIONS", f"{WORK_DIR}/descriptions/descriptions.jsonl")
AUTODDG_SPEC = "autoddg @ https://github.com/VIDA-NYU/AutoDDG/archive/f286ac697bdb0a199eac1ce0778401f30341b4bb.zip"

# Which benchmark to score:
#   "acquired": queries/pools restricted to tables that were read successfully (notebook 01, section 6) - use this to
#               compare Original vs UFD vs SFD on identical documents
#   "planned" : every judged CSV candidate (Original only; available before any table is downloaded)
#   "auto"    : "acquired" if it exists, else "planned"
POOL_SET = "auto"
KS = [5, 10, 15, 20]
# Fields indexed per document. AutoDDG's SimpleSearchEngine default is title + description; the paper does not say.
INDEX_CONFIGS = {
    "title+description": {"title": 1, "description": 1},
    "description only": {"description": 1},
}
VARIANTS = ["Original", "UFD", "SFD"]  # UFD/SFD are skipped automatically if descriptions.jsonl is missing
MISSING_DESCRIPTION = "empty"  # a doc with no generated text: "empty" (index title only / nothing) or "original"
RANDOM_BASELINE_SHUFFLES = 200
PAPER = {"Original": 0.688, "SFD": 0.705}  # NDCG@20, BM25

if os.environ.get("NTCIR_DDG_OVERRIDES"):
    import json as _json

    globals().update(_json.loads(os.environ["NTCIR_DDG_OVERRIDES"]))

In [ ]:
import sys
import subprocess

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB and WORK_DIR.startswith("/content/drive"):
    from google.colab import drive

    drive.mount("/content/drive")
import warnings

warnings.filterwarnings("ignore", message="transformers and torch not available")
try:
    import autoddg  # noqa: F401
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", AUTODDG_SPEC], check=True)

# AutoDDG's SimpleSearchEngine downloads NLTK "punkt", but nltk>=3.9 tokenizes with "punkt_tab": fetch both.
import nltk  # noqa: E402

for _res in ("punkt", "punkt_tab", "stopwords"):
    nltk.download(_res, quiet=True)

In [ ]:
import collections
import json
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", message="transformers and torch not available")
from autoddg.ranking import compute_ndcg  # noqa: E402
from autoddg.search import SimpleSearchEngine  # noqa: E402

WORK = Path(WORK_DIR)
BENCH, RESULTS = WORK / "benchmark", WORK / "results"
RESULTS.mkdir(parents=True, exist_ok=True)


def read_jsonl(path):
    rows = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                try:
                    rows.append(json.loads(line))
                except json.JSONDecodeError:
                    pass
    return rows


def read_tsv_queries(path):
    return dict(line.rstrip("\n").split("\t", 1) for line in open(path, encoding="utf-8") if line.strip())


def read_tsv_qrels(path):
    q = collections.defaultdict(dict)
    for line in open(path, encoding="utf-8"):
        if line.strip():
            qid, doc, rel = line.rstrip("\n").split("\t")
            q[qid][doc] = int(rel)
    return dict(q)


pool_set = POOL_SET
if pool_set == "auto":
    pool_set = "acquired" if (BENCH / "pools_acquired.json").exists() else "planned"
suffix = "_acquired" if pool_set == "acquired" else ""
queries = read_tsv_queries(BENCH / f"queries{suffix}.tsv")
qrels = read_tsv_qrels(BENCH / f"qrels{suffix}.tsv")
pools = json.loads((BENCH / f"pools{suffix}.json").read_text())
docs = {r["id"]: r for r in read_jsonl(BENCH / "docs.jsonl")}
config = json.loads((BENCH / "config.json").read_text())
print(f"benchmark: {pool_set}  config={config['chosen']}")
print(f"{len(queries)} queries, {sum(len(p) for p in pools.values())} (query, doc) pairs, "
      f"{len({d for p in pools.values() for d in p})} unique docs")
missing = [d for p in pools.values() for d in p if d not in docs]
assert not missing, f"{len(missing)} pool docs missing from docs.jsonl"

## Load generated descriptions (if any)
`descriptions.jsonl` may contain several rows per id (re-runs append); the last row without an error wins. All rows
must come from **one pinned model**: we stop if several model ids are present.

In [ ]:
gen = {}
models = collections.Counter()
if Path(DESCRIPTIONS_PATH).exists():
    for row in read_jsonl(DESCRIPTIONS_PATH):
        if row.get("error") or not row.get("ufd"):
            continue
        gen[row["id"]] = row
    models = collections.Counter(r["model"] for r in gen.values())
    print(f"descriptions: {len(gen)} tables with UFD/SFD, models: {dict(models)}")
    if len(models) > 1:
        raise ValueError(f"descriptions.jsonl mixes models {dict(models)}; all experiments must use one pinned model")
    pool_docs = {d for p in pools.values() for d in p}
    print(f"coverage of the evaluated pools: {len(pool_docs & set(gen))}/{len(pool_docs)}")
else:
    print(f"no descriptions at {DESCRIPTIONS_PATH} -> evaluating Original only")
variants = [v for v in VARIANTS if v == "Original" or gen]


def text_for(variant, doc_id):
    if variant == "Original":
        return docs[doc_id].get("description") or ""
    row = gen.get(doc_id)
    if row is None:
        return (docs[doc_id].get("description") or "") if MISSING_DESCRIPTION == "original" else ""
    return row["ufd"] if variant == "UFD" else row["sfd"]

## Evaluate
Documents are indexed in **sorted doc-id order**. `SimpleSearchEngine.search` uses a stable sort, so tied BM25 scores
(e.g. documents sharing no term with the query) keep insertion order; sorted ids are arbitrary with respect to
relevance, so ties neither help nor hurt. Indexing documents grouped by relevance would silently inflate NDCG.

In [ ]:
def evaluate(variant, index_fields):
    rows = []
    for qid in sorted(queries):
        pool = sorted(pools[qid])
        engine = SimpleSearchEngine("BM25")
        engine.index_documents(
            {d: {"title": docs[d].get("title") or "", "description": text_for(variant, d), "qrel": str(qrels[qid][d])}
             for d in pool},
            index_fields=index_fields)
        retrieved, ideal, ranked = engine.search(queries[qid], size=len(pool))
        row = {"qid": qid, "query": queries[qid], "n_docs": len(pool), "n_relevant": sum(v > 0 for v in qrels[qid].values())}
        for k in KS:
            row[f"ndcg@{k}"] = compute_ndcg(retrieved, ideal, k)
        rows.append(row)
    return rows


per_query = []
for variant in variants:
    for name, fields in INDEX_CONFIGS.items():
        for r in evaluate(variant, fields):
            per_query.append({"variant": variant, "index": name, **r})
per_query_df = pd.DataFrame(per_query)


def random_baseline(k_list, shuffles):
    rng = random.Random(0)
    out = {f"ndcg@{k}": [] for k in k_list}
    for qid in sorted(queries):
        rels = [float(qrels[qid][d]) for d in pools[qid]]
        ideal = sorted(rels, reverse=True)
        for k in k_list:
            vals = []
            for _ in range(shuffles):
                rng.shuffle(rels)
                vals.append(compute_ndcg(rels, ideal, k))
            out[f"ndcg@{k}"].append(np.mean(vals))
    return {k: float(np.mean(v)) for k, v in out.items()}


cols = [f"ndcg@{k}" for k in KS]
summary = per_query_df.groupby(["variant", "index"], sort=False)[cols].mean().reset_index()
summary["n_queries"] = per_query_df.groupby(["variant", "index"], sort=False).size().values
rb = random_baseline(KS, RANDOM_BASELINE_SHUFFLES)
summary = pd.concat([summary, pd.DataFrame([{"variant": "random ranking", "index": "-", **rb,
                                             "n_queries": len(queries)}])], ignore_index=True)
summary["paper ndcg@20"] = summary.variant.map(PAPER)
display(summary.round(3))

tag = f"{pool_set}" + (f"_{next(iter(models))}" if models else "")
summary.to_csv(RESULTS / f"summary_{tag}.csv", index=False)
per_query_df.to_csv(RESULTS / f"per_query_{tag}.csv", index=False)
print(f"saved {RESULTS / f'summary_{tag}.csv'} and per_query_{tag}.csv")

### Per-query view
Wide table of NDCG@20 per query, useful for spotting queries where generated descriptions help or hurt.

In [ ]:
wide = per_query_df.pivot_table(index=["qid", "query"], columns=["variant", "index"], values="ndcg@20")
display(wide.round(3))

## SPLADE (stub)

The paper also reports a SPLADE-style sparse retriever. AutoDDG supports it through
`SimpleSearchEngine("COSINE")`, which expects **precomputed FastEmbed sparse vectors** (objects with `.indices` and
`.values`) instead of text: the first enabled field of `index_fields` holds the document vector, and `search()`
receives the query vector.

```python
from fastembed import SparseTextEmbedding
model = SparseTextEmbedding("prithivida/Splade_PP_en_v1")
doc_vecs = dict(zip(ids, model.embed([title + " " + text for ...])))
q_vec = next(model.query_embed(query))
engine = SimpleSearchEngine("COSINE")
engine.index_documents({d: {"embedding": doc_vecs[d], "qrel": str(rel)} for d in pool},
                       index_fields={"embedding": 1})
retrieved, ideal, _ = engine.search(q_vec, size=len(pool))
```

Not run here (needs a ~500 MB model download). `index_documents` is annotated `dict[str, dict[str, str]]` and
AutoDDG uses beartype, so passing vector objects may need beartype disabled for that call; check before relying on it.

In [ ]:
RUN_SPLADE = False
if RUN_SPLADE:
    raise NotImplementedError("SPLADE evaluation is a stub; see the markdown cell above.")